69 mins 35.7 secs

## Libraries

In [34]:
import numpy as np
import pandas as pd
import time
import random

from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import ParameterGrid

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_

## Config

In [35]:
# ---------------- CONFIG ----------------
DATA_PATH = "../../data/full_data.xlsx"

TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

# Tuning window (no test here: just 2007-04 to 2022-03)
TUNE_START_DATE = pd.Timestamp("2007-04-01")
TUNE_END_DATE   = pd.Timestamp("2022-03-31")

# Rolling CV: 10 years train, 1 year val (in months)
ROLLING_TRAIN_WINDOW = 90   # 90 months = 7.5 years
ROLLING_VAL_WINDOW   = 18    # 18 months = 1.5 years

# Hyperparameter grid for CNN-LSTM
param_grid = list(ParameterGrid({
    "WINDOW":       [12, 18],    # input sequence length (months)
    "CNN_CHANNELS": [16, 32],
    "KERNEL_SIZE":  [3, 5],
    "LSTM_HIDDEN":  [32, 64],
    "DROPOUT":      [0.0, 0.3],
    "LR":           [1e-3, 5e-4],
    "WEIGHT_DECAY": [0.0, 1e-4],
}))

print(f"Number of hyperparameter configs: {len(param_grid)}")

BATCH_SIZE    = 64
MAX_EPOCHS    = 80
PATIENCE      = 5
MAX_GRAD_NORM = 5.0

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Feature lists
continuous_cols = [
    "AverageNeighbourPrice",
    "local_I",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
]

categorical_cols = [
    "LMIQuadrant__2",
    "LMIQuadrant__3",
    "LMIQuadrant__4",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

base_feature_cols = continuous_cols + categorical_cols

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



Number of hyperparameter configs: 128
Using device: cuda


## Metric functions

In [36]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(
        100.0 * np.mean(
            2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps)
        )
    )

def mase(y, yhat, y_train, m=12, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)

    if len(y_train) <= m:
        return np.nan

    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return float(np.mean(np.abs(y - yhat)) / scale)


## Load and prepare panel data

In [37]:
df = pd.read_excel(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([TIME_COL, ENTITY_COL]).reset_index(drop=True)

# restrict to tuning period
mask_tune = (df[TIME_COL] >= TUNE_START_DATE) & (df[TIME_COL] <= TUNE_END_DATE)
df = df.loc[mask_tune].copy()

# ensure 1 row per (Date, AreaCode)
df = (
    df.sort_values([TIME_COL, ENTITY_COL])
      .drop_duplicates(subset=[TIME_COL, ENTITY_COL], keep="last")
      .copy()
)

# Build complete panel [T, N]
dates = pd.Index(sorted(df[TIME_COL].unique()))
T_total = len(dates)
la_order = sorted(df[ENTITY_COL].unique())
N = len(la_order)

print(f"Tuning period: {dates[0].date()} → {dates[-1].date()}")
print("Total months:", T_total)
print("Number of LAs:", N)

full_index = pd.MultiIndex.from_product(
    [dates, la_order],
    names=[TIME_COL, ENTITY_COL]
)

feature_cols = base_feature_cols.copy()

df_panel = (
    df.set_index([TIME_COL, ENTITY_COL])
      .reindex(full_index)
      .sort_index()
)

# Forward/backward fill features + target within each LA
df_panel[feature_cols + [TARGET_COL]] = (
    df_panel[feature_cols + [TARGET_COL]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

Tuning period: 2007-04-01 → 2022-03-01
Total months: 180
Number of LAs: 294


## Add lagged price features (1 and 12 months)

In [38]:
df_panel["price_lag1"] = (
    df_panel
    .groupby(level=ENTITY_COL)[TARGET_COL]
    .shift(1)
)

df_panel["price_lag12"] = (
    df_panel
    .groupby(level=ENTITY_COL)[TARGET_COL]
    .shift(12)
)

# fill lags within each LA so we don't introduce NaNs
df_panel[["price_lag1", "price_lag12"]] = (
    df_panel[["price_lag1", "price_lag12"]]
        .groupby(level=ENTITY_COL)
        .ffill()
        .bfill()
)

lag_price_cols = ["price_lag1", "price_lag12"]

feature_cols = feature_cols + lag_price_cols

# last-resort fill for any remaining NaNs
missing_total = df_panel[feature_cols + [TARGET_COL]].isna().sum().sum()
if missing_total > 0:
    print(f"⚠ {missing_total} NaNs after lag creation. Filling with column means.")
    col_means = df_panel[feature_cols + [TARGET_COL]].mean()
    df_panel[feature_cols + [TARGET_COL]] = df_panel[feature_cols + [TARGET_COL]].fillna(col_means)


⚠ 1 NaNs after lag creation. Filling with column means.


## Build X_all, y_all arrays

In [39]:
F = len(feature_cols)

X_all = (
    df_panel[feature_cols]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N, F)
)
y_all = (
    df_panel[TARGET_COL]
    .to_numpy(dtype=np.float32)
    .reshape(T_total, N)
)

print("X_all shape:", X_all.shape)  # [T, N, F]
print("y_all shape:", y_all.shape)  # [T, N]

# keep original y for RMSE on original scale
y_all_orig = y_all.copy()


X_all shape: (180, 294, 31)
y_all shape: (180, 294)


## Rolling origin folds over time

In [40]:
fold_specs = []
start_idx = ROLLING_TRAIN_WINDOW   # first index where a val window can start

while True:
    train_end_idx = start_idx            # exclusive
    val_start_idx = train_end_idx
    val_end_idx   = val_start_idx + ROLLING_VAL_WINDOW
    if val_end_idx > T_total:
        break

    train_start_idx = train_end_idx - ROLLING_TRAIN_WINDOW

    train_start_date = dates[train_start_idx]
    train_end_date   = dates[train_end_idx - 1]
    val_start_date   = dates[val_start_idx]
    val_end_date     = dates[val_end_idx - 1]

    fold_specs.append(
        (train_start_idx, train_end_idx, val_start_idx, val_end_idx,
         train_start_date, train_end_date, val_start_date, val_end_date)
    )

    start_idx += ROLLING_VAL_WINDOW

print(f"Number of folds: {len(fold_specs)}")
for i, (tr_s, tr_e, v_s, v_e, ts, te, vs, ve) in enumerate(fold_specs, start=1):
    print(f"  Fold {i}: Train {ts:%Y-%m}–{te:%Y-%m}, Val {vs:%Y-%m}–{ve:%Y-%m}")


Number of folds: 5
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
  Fold 2: Train 2008-10–2016-03, Val 2016-04–2017-09
  Fold 3: Train 2010-04–2017-09, Val 2017-10–2019-03
  Fold 4: Train 2011-10–2019-03, Val 2019-04–2020-09
  Fold 5: Train 2013-04–2020-09, Val 2020-10–2022-03


## Data for CNN-LSTM

In [44]:
class PanelWindowDataset(Dataset):
    """
    Each sample is (X_seq, y_t) for a specific LA and time t, in a *local* window:
      - X_seq: [window, F]
      - y_t: scalar
    X and y are fold-specific arrays: [T_fold, N, F] and [T_fold, N].
    """
    def __init__(self, X, y, window):
        self.X = X
        self.y = y
        self.window = window
        self.T, self.N, self.F = X.shape

        indices = []
        for t in range(window, self.T):
            for n in range(self.N):
                indices.append((t, n))
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        t, n = self.indices[idx]
        X_seq = self.X[t - self.window:t, n, :]  # [window, F]
        y_t   = self.y[t, n]                     # scalar
        return (
            torch.tensor(X_seq, dtype=torch.float32),
            torch.tensor(y_t,   dtype=torch.float32),
        )

## CNN-LSTM model

In [45]:
class CNNLSTM(nn.Module):
    def __init__(self, in_feats, cnn_channels, kernel_size, lstm_hidden, dropout=0.0):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels=in_feats,
            out_channels=cnn_channels,
            kernel_size=kernel_size,
            padding="same"
        )
        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden, 1)

    def forward(self, X_seq):
        """
        X_seq: [B, T, F]
        """
        B, T, F = X_seq.shape
        # Conv1d expects [B, C_in, T]
        x = X_seq.permute(0, 2, 1)   # [B, F, T]
        x = self.conv(x)             # [B, C_out, T]
        x = torch.relu(x)
        # back to [B, T, C_out]
        x = x.permute(0, 2, 1)       # [B, T, C_out]
        out, (h_n, c_n) = self.lstm(x)  # out: [B, T, H], h_n: [1, B, H]
        h_last = h_n[-1]             # [B, H]
        h_last = self.dropout(h_last)
        y_hat = self.fc(h_last).squeeze(-1)  # [B]
        return y_hat

## Tuning

In [46]:
results = []

for cfg_id, params in enumerate(param_grid, start=1):
    WINDOW       = params["WINDOW"]
    CNN_CHANNELS = params["CNN_CHANNELS"]
    KERNEL_SIZE  = params["KERNEL_SIZE"]
    LSTM_HIDDEN  = params["LSTM_HIDDEN"]
    DROPOUT      = params["DROPOUT"]
    LR           = params["LR"]
    WD           = params["WEIGHT_DECAY"]

    print(f"\n=== Config {cfg_id}/{len(param_grid)} ===")
    print(params)

    fold_mae_list   = []
    fold_rmse_list  = []
    fold_smape_list = []
    fold_mase_list  = []

    fold_count = 0

    for fold_no, (train_start_idx, train_end_idx,
                  val_start_idx, val_end_idx,
                  train_start_date, train_end_date,
                  val_start_date, val_end_date) in enumerate(fold_specs, start=1):

        print(f"  Fold {fold_no}: Train {train_start_date:%Y-%m}–{train_end_date:%Y-%m}, "
              f"Val {val_start_date:%Y-%m}–{val_end_date:%Y-%m}")

        # -------- LEAK-FREE SCALING FOR THIS FOLD ONLY --------
        X_train = X_all[train_start_idx:train_end_idx]   # [T_tr, N, F]
        X_val   = X_all[val_start_idx:val_end_idx]       # [T_val, N, F]

        y_train = y_all[train_start_idx:train_end_idx]   # [T_tr, N]
        y_val   = y_all[val_start_idx:val_end_idx]       # [T_val, N]

        if X_train.shape[0] <= WINDOW or X_val.shape[0] <= WINDOW:
            print("    (skip fold: not enough time steps after WINDOW constraint)")
            continue

        x_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(
            X_train.reshape(-1, F)
        ).reshape(X_train.shape)
        X_val_scaled = x_scaler.transform(
            X_val.reshape(-1, F)
        ).reshape(X_val.shape)

        y_scaler = RobustScaler()
        y_train_scaled = y_scaler.fit_transform(
            y_train.reshape(-1, 1)
        ).reshape(y_train.shape)
        y_val_scaled = y_scaler.transform(
            y_val.reshape(-1, 1)
        ).reshape(y_val.shape)

        y_scale_factor = float(y_scaler.scale_[0])

        # build datasets & loaders on local (train/val) arrays
        train_ds = PanelWindowDataset(X_train_scaled, y_train_scaled, window=WINDOW)
        val_ds   = PanelWindowDataset(X_val_scaled,   y_val_scaled,   window=WINDOW)

        if len(train_ds) == 0 or len(val_ds) == 0:
            print("    (skip fold: empty dataset after windowing)")
            continue

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = CNNLSTM(
            in_feats=F,
            cnn_channels=CNN_CHANNELS,
            kernel_size=KERNEL_SIZE,
            lstm_hidden=LSTM_HIDDEN,
            dropout=DROPOUT
        ).to(DEVICE)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
        loss_fn = nn.MSELoss()

        best_val_mse = np.inf
        best_epoch = -1
        epochs_no_improve = 0
        best_state = None

        # --------------- TRAIN + EARLY STOP ---------------
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_losses = []

            for X_seq, y_t in train_loader:
                X_seq = X_seq.to(DEVICE)  # [B, T, F]
                y_t   = y_t.to(DEVICE)    # [B]

                optimizer.zero_grad()
                y_hat = model(X_seq)
                loss  = loss_fn(y_hat, y_t)

                if not torch.isfinite(loss):
                    print(f"    ⚠ Non-finite training loss at epoch {epoch}. Skipping fold.")
                    train_losses = []
                    break

                loss.backward()
                clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
                optimizer.step()
                train_losses.append(loss.item())

            if not train_losses:
                break

            # validation
            model.eval()
            val_losses = []
            with torch.no_grad():
                for X_seq, y_t in val_loader:
                    X_seq = X_seq.to(DEVICE)
                    y_t   = y_t.to(DEVICE)
                    y_hat = model(X_seq)
                    vloss = loss_fn(y_hat, y_t)
                    if torch.isfinite(vloss):
                        val_losses.append(vloss.item())

            if not val_losses:
                print("    ⚠ All val losses non-finite. Skipping fold.")
                break

            val_mse = float(np.mean(val_losses))
            val_rmse_orig = np.sqrt(val_mse) * y_scale_factor

            print(f"    Epoch {epoch:03d} | "
                  f"train MSE={np.mean(train_losses):.4f} | "
                  f"val MSE={val_mse:.4f} | "
                  f"val RMSE(£)≈{val_rmse_orig:,.1f}")

            if val_mse + 1e-6 < best_val_mse:
                best_val_mse = val_mse
                best_epoch   = epoch
                epochs_no_improve = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    print(f"    Early stopping at epoch {epoch}")
                    break

        if best_epoch == -1 or best_state is None:
            print("    ❌ Fold failed (no valid epoch). Skipping fold.")
            continue

        # restore best weights
        model.load_state_dict(best_state)

        # --------------- FINAL VAL METRICS FOR THIS FOLD ---------------
        model.eval()
        y_true_val_scaled = []
        y_pred_val_scaled = []

        with torch.no_grad():
            for X_seq, y_t in val_loader:
                X_seq = X_seq.to(DEVICE)
                y_t   = y_t.to(DEVICE)
                y_hat = model(X_seq)
                y_true_val_scaled.append(y_t.cpu().numpy())
                y_pred_val_scaled.append(y_hat.cpu().numpy())

        y_true_val_scaled = np.concatenate(y_true_val_scaled, axis=0).reshape(-1, 1)
        y_pred_val_scaled = np.concatenate(y_pred_val_scaled, axis=0).reshape(-1, 1)

        y_true_val_orig = y_scaler.inverse_transform(y_true_val_scaled).ravel()
        y_pred_val_orig = y_scaler.inverse_transform(y_pred_val_scaled).ravel()

        # MASE scaling: use TRAIN window in original scale
        y_train_fold_orig = y_all_orig[train_start_idx:train_end_idx].reshape(-1)

        fold_mae   = mae(y_true_val_orig, y_pred_val_orig)
        fold_rmse  = rmse(y_true_val_orig, y_pred_val_orig)
        fold_smape = smape(y_true_val_orig, y_pred_val_orig)
        fold_mase  = mase(y_true_val_orig, y_pred_val_orig, y_train_fold_orig, m=12)

        print(f"    Fold {fold_no} MAE(£)={fold_mae:,.1f}, "
              f"RMSE(£)={fold_rmse:,.1f}, sMAPE={fold_smape:.3f}%, MASE={fold_mase:.3f}")

        fold_mae_list.append(fold_mae)
        fold_rmse_list.append(fold_rmse)
        fold_smape_list.append(fold_smape)
        fold_mase_list.append(fold_mase)
        fold_count += 1

    if fold_count == 0:
        print("  ❌ No valid folds for this config. Skipping.")
        continue

    cfg_result = {
        "model_type": "CNNLSTM",
        "WINDOW": WINDOW,
        "CNN_CHANNELS": CNN_CHANNELS,
        "KERNEL_SIZE": KERNEL_SIZE,
        "LSTM_HIDDEN": LSTM_HIDDEN,
        "DROPOUT": DROPOUT,
        "LR": LR,
        "WEIGHT_DECAY": WD,
        "folds_used": fold_count,

        "MAE_mean":   float(np.mean(fold_mae_list)),
        "MAE_std":    float(np.std(fold_mae_list)),
        "RMSE_mean":  float(np.mean(fold_rmse_list)),
        "RMSE_std":   float(np.std(fold_rmse_list)),
        "sMAPE_mean": float(np.mean(fold_smape_list)),
        "sMAPE_std":  float(np.std(fold_smape_list)),
        "MASE_mean":  float(np.mean(fold_mase_list)),
        "MASE_std":   float(np.std(fold_mase_list)),
    }
    results.append(cfg_result)


=== Config 1/128 ===
{'CNN_CHANNELS': 16, 'DROPOUT': 0.0, 'KERNEL_SIZE': 3, 'LR': 0.001, 'LSTM_HIDDEN': 32, 'WEIGHT_DECAY': 0.0, 'WINDOW': 12}
  Fold 1: Train 2007-04–2014-09, Val 2014-10–2016-03
    Epoch 001 | train MSE=0.1469 | val MSE=0.2619 | val RMSE(£)≈52,004.7
    Epoch 002 | train MSE=0.0273 | val MSE=0.1754 | val RMSE(£)≈42,553.7
    Epoch 003 | train MSE=0.0150 | val MSE=0.1205 | val RMSE(£)≈35,276.7
    Epoch 004 | train MSE=0.0088 | val MSE=0.0462 | val RMSE(£)≈21,837.8
    Epoch 005 | train MSE=0.0062 | val MSE=0.0451 | val RMSE(£)≈21,588.7
    Epoch 006 | train MSE=0.0057 | val MSE=0.0632 | val RMSE(£)≈25,551.4
    Epoch 007 | train MSE=0.0049 | val MSE=0.0557 | val RMSE(£)≈23,982.3
    Epoch 008 | train MSE=0.0049 | val MSE=0.0445 | val RMSE(£)≈21,446.2
    Epoch 009 | train MSE=0.0045 | val MSE=0.0484 | val RMSE(£)≈22,349.5
    Epoch 010 | train MSE=0.0047 | val MSE=0.0439 | val RMSE(£)≈21,291.0
    Epoch 011 | train MSE=0.0043 | val MSE=0.0335 | val RMSE(£)≈18,606.4


## Results

In [47]:

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("RMSE_mean").reset_index(drop=True)
    print("\n=== TOP CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===")
    print(results_df.head(10))
    results_df.to_csv("../../results/cnnlstm_rollingcv_results.csv", index=False)
    print("\nSaved tuning results to ../../results/cnnlstm_rollingcv_results.csv")
else:
    print("\nNo successful configs to report.")



=== TOP CONFIGS BY MEAN VAL RMSE (ORIGINAL £) ===
  model_type  WINDOW  CNN_CHANNELS  KERNEL_SIZE  LSTM_HIDDEN  DROPOUT      LR  \
0    CNNLSTM      12            32            5           64      0.3  0.0005   
1    CNNLSTM      12            16            3           64      0.0  0.0005   
2    CNNLSTM      12            32            5           64      0.0  0.0010   
3    CNNLSTM      12            16            3           64      0.0  0.0010   
4    CNNLSTM      12            16            5           64      0.3  0.0010   
5    CNNLSTM      12            16            3           64      0.3  0.0005   
6    CNNLSTM      12            16            5           64      0.3  0.0010   
7    CNNLSTM      12            32            3           64      0.0  0.0005   
8    CNNLSTM      12            16            3           64      0.3  0.0010   
9    CNNLSTM      12            32            3           64      0.3  0.0005   

   WEIGHT_DECAY  folds_used      MAE_mean      MAE_std   